# Pipeline CPU n°2 : PyTorch → TorchScript / torch.compile
TP1 Déploiement IA — ResNet-18 / CIFAR-10. **Environnement d'exécution Colab : CPU (Exécution > Modifier le type d'exécution > CPU).**

| Niveau | Composant |
|---|---|
| Haut niveau | PyTorch (ResNet-18 pré-entraîné) |
| Format | TorchScript (trace + freeze) et graphe FX PyTorch 2 (TorchDynamo) |
| Middle level | optimisations de graphe JIT / compilateur TorchInductor |
| Bas niveau | kernels ATen/oneDNN + code C++/OpenMP généré par Inductor |
| Matériel | CPU Google Colab |

[Ouvrir dans Colab](https://colab.research.google.com/github/Romain-LM-Dev/TP1_deploiement_IA/blob/main/notebooks/02_cpu_torchcompile.ipynb)

## Protocole (identique aux 4 pipelines, défini dans `common.py`)
- **Données** : les 50 premières images du jeu de test CIFAR-10, redimensionnées de 32×32 à 224×224 (bilinéaire) puis normalisées avec la moyenne/écart-type ImageNet.
- **Modèle** : ResNet-18 pré-entraîné ImageNet (`IMAGENET1K_V1`). Les 1000 classes sont regroupées en 10 classes CIFAR (ex. les 118 races de chiens → *dog*) ; la classe prédite est celle dont le meilleur logit est le plus élevé.
- **Mesure** : batchs de 10 images, données déjà sur le périphérique, 1 appel initial (compilation éventuelle, chronométré à part), 3 passes de chauffe, puis 10 exécutions mesurées des 50 images. Synchronisation explicite après chaque batch sur GPU.
- **Énergie** : compteur matériel (NVML sur GPU, RAPL sur CPU si accessible), intégré sur ≥ 3 s d'inférence en boucle après chaque exécution.
- **Référence** : PyTorch *eager* sur le même matériel, pour mesurer le gain et vérifier que les prédictions ne sont pas dégradées.

In [ ]:
# Récupération du dépôt : code commun (données, modèle, protocole de mesure)
import os, subprocess, sys
REPO = "https://github.com/Romain-LM-Dev/TP1_deploiement_IA.git"
if os.path.exists("../common.py"):          # exécution locale depuis notebooks/
    os.chdir("..")
elif not os.path.exists("common.py"):       # Google Colab
    if not os.path.exists("TP1_deploiement_IA"):
        subprocess.run(["git", "clone", "-q", REPO], check=True)
    os.chdir("TP1_deploiement_IA")
sys.path.insert(0, os.getcwd())
import common as C
C.env_info()

In [ ]:
import time
from collections import Counter
import numpy as np, torch
from IPython.display import display

torch.manual_seed(C.SEED)
x, y = C.load_data()            # (50, 3, 224, 224), normalisation ImageNet
model = C.load_model()          # ResNet-18 IMAGENET1K_V1, mode eval
print("Entrée :", tuple(x.shape), "| batch :", C.BATCH_SIZE, "| répétitions :", C.N_RUNS)
print("Répartition des classes :", dict(zip(C.CIFAR_CLASSES, np.bincount(y, minlength=10).tolist())))
results = []

In [ ]:
meter = C.EnergyMeter("cpu")
print("Source énergie :", meter.source)
batches = C.make_batches(x)

## 1. Référence : PyTorch eager sur CPU

In [ ]:
with torch.inference_mode():
    eager = C.benchmark("CPU-2 PyTorch", "PyTorch eager (réf.)", model, batches, y, meter=meter)
ref = eager["_logits"]
results.append(eager)

## 2. TorchScript
`torch.jit.trace` capture le graphe, `freeze` transforme les poids en constantes et `optimize_for_inference` replie les BatchNorm dans les convolutions et fusionne les activations.

In [ ]:
with torch.inference_mode():
    t0 = time.perf_counter()
    traced = torch.jit.trace(model, batches[0])
    ts = torch.jit.optimize_for_inference(torch.jit.freeze(traced))
    ts_s = time.perf_counter() - t0
kinds = lambda g: Counter(n.kind() for n in g.nodes() if n.kind().startswith("aten::"))
print(f"Trace + optimisation : {ts_s:.2f} s")
print("Graphe tracé    :", dict(kinds(traced.inlined_graph)))
print("Graphe optimisé :", dict(kinds(ts.graph)))

In [ ]:
with torch.inference_mode():
    r = C.benchmark("CPU-2 PyTorch", "TorchScript (freeze)", ts, batches, y, ref=ref, meter=meter)
r["compile_s"] = ts_s
results.append(r)

## 3. torch.compile (TorchInductor)
TorchDynamo capture le graphe Python, TorchInductor le compile : les opérations élémentaires sont fusionnées en boucles C++/OpenMP vectorisées et les convolutions sont déléguées à oneDNN. `freezing=True` (recommandé pour l'inférence CPU) traite les poids comme des constantes, ce qui permet le repliement Conv+BN, comme pour TorchScript. La compilation a lieu au premier appel (`first_call_s`).

In [ ]:
import torch._dynamo, torch._inductor.config
torch._inductor.config.freezing = True
expl = torch._dynamo.explain(model)(batches[0])
print(f"Graphes capturés : {expl.graph_count} | ruptures de graphe : {expl.graph_break_count} | opérations : {expl.op_count}")
torch._dynamo.reset()
compiled = torch.compile(model)
with torch.inference_mode():
    r = C.benchmark("CPU-2 PyTorch", "torch.compile (inductor)", compiled, batches, y, ref=ref, meter=meter)
results.append(r)

## 4. Synthèse, graphiques et sauvegarde

In [ ]:
df = C.plot_summary(results, "Pipeline CPU n°2 : PyTorch → TorchScript / torch.compile", "figures/02_cpu_torchcompile.png")
display(df[C.TABLE_COLS].round(4).T)
path = C.save_results("02_cpu_torchcompile", results, {})

In [ ]:
# Colab dans le navigateur : télécharge le JSON (à déposer dans results/ du dépôt).
# Extension Colab de VS Code : rien à faire, il suffit d'enregistrer le notebook (résultats intégrés ci-dessus).
try:
    from google.colab import files
    files.download(path)
except Exception:
    pass

## Lecture des résultats
- **Débit / temps** : comparer chaque variante à la référence *eager* de ce notebook (même machine, même session).
- **Stabilité** : un coefficient de variation (`throughput_cv_pct`) < 5 % indique des mesures reproductibles ; au-delà, regarder le *boxplot* (valeurs aberrantes = interférences de la VM partagée).
- **Qualité** : `agreement_vs_eager` doit valoir 100 % et `max_abs_diff_vs_eager` rester faible (~1e-4 en FP32).
- L'analyse croisée des 4 pipelines est faite dans `05_comparaison.ipynb` et dans le rapport.